# Understanding why

## Replication of Modeling FTS

In [63]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd

from IPython.display import Image, display

In [64]:
sys.path.append(os.path.abspath(".."))

import replication.stylized_facts as sf
import numpy as np

In [65]:
# Folder containing the shared scripts:
# stylized_facts.py, stats.py, visualize.py
MIN_RECORDS = 1000
MAX_LAG_ACF = 1000
MAX_LAG_LEVERAGE = 100
DIST_BINS = 100

In [66]:
RAW_DIR = Path("../data/raw_replication")
RAW_OUTPUT_DIR = Path("./stylized_facts_output_raw")
RAW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_stock_returns = {}

for csv_path in sorted(RAW_DIR.glob("*.csv")):
    try:
        df = pd.read_csv(csv_path)
        df.columns = [c.lower().strip() for c in df.columns]

        if "date" not in df.columns or "adj_close" not in df.columns:
            print(f"Skipping {csv_path.name}: missing expected columns, found {list(df.columns)}")
            continue

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date", "adj_close"]).sort_values("date")
        df = df.drop_duplicates(subset=["date"], keep="last").reset_index(drop=True)

        df["log_return"] = np.log(df["adj_close"])
        df["log_return"] = df["log_return"].diff()
        df = df.dropna(subset=["log_return"]).reset_index(drop=True)

        r = df["log_return"].to_numpy(dtype=float)
        if len(r) > MIN_RECORDS:
            symbol = csv_path.stem
            raw_stock_returns[symbol] = r

    except Exception as e:
        print(f"Skipping {csv_path.name}: {e}")

print(f"Eligible symbols: {len(raw_stock_returns)}")
raw_summary = pd.DataFrame({
    "Symbol": list(raw_stock_returns.keys()),
    "n_returns": [len(v) for v in raw_stock_returns.values()],
}).sort_values("n_returns", ascending=False)
display(raw_summary.head())
display(raw_summary.describe(include="all"))
print(raw_summary.shape)


Eligible symbols: 210


,Symbol,n_returns
209,raw_XOM_1962-01-02_2026-04-10,16175
34,raw_CAT_1962-01-02_2026-04-10,16175
50,raw_CVX_1962-01-02_2026-04-10,16175
21,raw_BA_1962-01-02_2026-04-10,16175
55,raw_DIS_1962-01-02_2026-04-10,16175


,Symbol,n_returns
count,210,210.000000
unique,210,NaN
top,raw_XOM_1962-01-02_2026-04-10,NaN
freq,1,NaN
mean,NaN,12492.809524
std,NaN,1699.903952
min,NaN,10083.000000
25%,NaN,11493.250000
50%,NaN,11610.000000
75%,NaN,13396.000000


(210, 2)


In [67]:
print(raw_summary.describe())
print(raw_summary.shape)
print(raw_summary.columns)

          n_returns
count    210.000000
mean   12492.809524
std     1699.903952
min    10083.000000
25%    11493.250000
50%    11610.000000
75%    13396.000000
max    16175.000000
(210, 2)
Index(['Symbol', 'n_returns'], dtype='object')


# Return others

In [68]:
import numpy as np
import pandas as pd
from pathlib import Path
attempt = False
if attempt:
    input_folder = Path("../data/raw_replication")

    stock_returns = {}
    for csv_path in sorted(input_folder.glob("*.csv")):
        df = pd.read_csv(csv_path, parse_dates=["date"])
        df["log_return"] = np.log(df["adj_close"] / df["adj_close"].shift(1))
        df = df.dropna(subset=["log_return"])
        stock_returns[csv_path.stem] = df["log_return"].to_numpy(dtype=float)
else:
    input_folder = Path("../data/replication_returns_other")
    stock_returns = {}
    for csv_path in sorted(input_folder.glob("*.csv")):
        df = pd.read_csv(csv_path, parse_dates=["date"])
        # df["log_return"] = np.log(df["adj_close"] / df["adj_close"].shift(1))
        # df = df.dropna(subset=["log_return"])
        stock_returns[csv_path.stem] = df["log_adj_close"].to_numpy(dtype=float)

    print(f"Loaded {len(stock_returns)} symbols")



Loaded 210 symbols


In [69]:
concat_stock_returns = np.concatenate(list(stock_returns.values()))

In [70]:
import numpy as np
import pandas as pd
from pathlib import Path

# # Recompute returns using processing_log_adj_close logic
# stock_returns = {}
# for csv_path in sorted(Path("../data/raw_replication").glob("*.csv")):
#     df = pd.read_csv(csv_path, parse_dates=["date"])
#     df["log_return"] = np.log(df["adj_close"] / df["adj_close"].shift(1))
#     df = df.dropna(subset=["log_return"])
#     stock_returns[csv_path.stem] = df["log_return"].to_numpy(dtype=float)

# Aggregate statistics — same analysis for both
def summary(returns_dict):
    flat = np.concatenate(list(returns_dict.values()))
    return pd.Series({
        "n_symbols" : len(returns_dict),
        "n_returns" : len(flat),
        "mean"      : flat.mean(),
        "std"       : flat.std(),
        "min"       : flat.min(),
        "q01"       : np.quantile(flat, 0.01),
        "q25"       : np.quantile(flat, 0.25),
        "median"    : np.median(flat),
        "q75"       : np.quantile(flat, 0.75),
        "q99"       : np.quantile(flat, 0.99),
        "max"       : flat.max(),
    })

pd.DataFrame({
    "raw_stock_returns" : summary(raw_stock_returns),
    "stock_returns"     : summary(stock_returns),
}).round(8)

,raw_stock_returns,stock_returns
n_symbols,2.100000e+02,2.100000e+02
n_returns,2.623490e+06,2.623490e+06
mean,4.514000e-04,4.514000e-04
std,2.086653e-02,2.086653e-02
min,-9.362579e-01,-9.362579e-01
q01,-5.640440e-02,-5.640440e-02
q25,-8.625820e-03,-8.625820e-03
median,0.000000e+00,0.000000e+00
q75,9.486230e-03,9.486230e-03
q99,5.849575e-02,5.849575e-02


In [71]:
a = np.sort(np.concatenate(list(stock_returns.values())))
b = np.sort(np.concatenate(list(raw_stock_returns.values())))

print(f"stock_returns    : {len(a)} values")
print(f"raw_stock_returns: {len(b)} values")

if len(a) == len(b):
    diff = np.abs(a - b)
    print(f"max |diff| : {diff.max():.2e}")
    print(f"mean |diff|: {diff.mean():.2e}")
    print(f"identical  : {(diff == 0).all()}")
else:
    print(f"Length mismatch: {len(a) - len(b):+d} — cannot do 1-to-1 comparison")

stock_returns    : 2623490 values
raw_stock_returns: 2623490 values
max |diff| : 1.06e-15
mean |diff|: 1.43e-16
identical  : False


In [72]:
# The sorted comparison proves the VALUES are the same but destroys temporal order.
# sf.acf and sf.leverage_effect are order-dependent — check per-stock ordering.

common = sorted(set(stock_returns) & set(raw_stock_returns))
print(f"Num stocks raw_returns: {len(raw_stock_returns)}, stock returns: {len(stock_returns)}, stocks in common: {len(common)}")

mismatches = []
for sym in common:
    a = stock_returns[sym]
    b = raw_stock_returns[sym]
    if len(a) != len(b):
        mismatches.append((sym, "length_diff", len(a) - len(b)))
    elif not np.allclose(a, b, atol=1e-12):
        max_diff = np.abs(a - b).max()
        mismatches.append((sym, "order_diff", max_diff))

if not mismatches:
    print("All per-stock series are identical in length and temporal order.")
else:
    print(f"{len(mismatches)} stocks differ:\n")
    for sym, kind, val in mismatches[:10]:
        print(f"  {sym}: {kind} = {val}")

Num stocks raw_returns: 210, stock returns: 210, stocks in common: 210
All per-stock series are identical in length and temporal order.


# Comparison

In [73]:
# Object array: one element per stock, each element is a 1D numpy array of log-returns.
# Used by sf.acf and sf.leverage_effect (multiple=True path).
raw_returns_obj = np.empty(len(raw_stock_returns), dtype=object)
for i, r in enumerate(raw_stock_returns.values()):
    raw_returns_obj[i] = r

# Flat concatenated array: all returns pooled into one 1D array.
# Used by sf.distribution (multiple=False path).
raw_pooled_returns = np.concatenate(list(raw_stock_returns.values()), axis=0)

print("raw_returns_obj shape:", raw_returns_obj.shape, "| dtype:", raw_returns_obj.dtype)
print("raw_pooled_returns_PROCESSED shape:", raw_pooled_returns.shape)
print("Number of zero (0.0%) returns:", sum(raw_pooled_returns == 0.0), f"({sum(raw_pooled_returns == 0) / raw_pooled_returns.size:.4%})")
print(f"Positive returns: {(raw_pooled_returns > 0).sum()}, Proportion: {((raw_pooled_returns > 0).sum() / raw_pooled_returns.size):.4%}")
print(f"Zero returns:     {(raw_pooled_returns == 0).sum()}, Proportion: {((raw_pooled_returns == 0).sum() / raw_pooled_returns.size):.4%}")
print(f"Negative returns: {(raw_pooled_returns < 0).sum()}, Proportion: {((raw_pooled_returns < 0).sum() / raw_pooled_returns.size):.4%}")


raw_returns_obj shape: (210,) | dtype: object
raw_pooled_returns_PROCESSED shape: (2623490,)
Number of zero (0.0%) returns: 200511 (7.6429%)
Positive returns: 1243975, Proportion: 47.4168%
Zero returns:     200511, Proportion: 7.6429%
Negative returns: 1179004, Proportion: 44.9403%


In [74]:
raw_returns_obj_PROCESSED = np.empty(len(stock_returns), dtype=object)
for i, r in enumerate(stock_returns.values()):
    raw_returns_obj_PROCESSED[i] = r

# Flat concatenated array: all returns pooled into one 1D array.
# Used by sf.distribution (multiple=False path).
raw_pooled_returns_PROCESSED = np.concatenate(list(stock_returns.values()), axis=0)

print("raw_returns_obj shape:", raw_returns_obj_PROCESSED.shape, "| dtype:", raw_returns_obj_PROCESSED.dtype)
print("raw_pooled_returns shape:", raw_pooled_returns_PROCESSED.shape)
print("Number of zero (0.0%) returns:", sum(raw_pooled_returns_PROCESSED == 0.0), f"({sum(raw_pooled_returns_PROCESSED == 0) / raw_pooled_returns_PROCESSED.size:.4%})")
print(f"Positive returns: {(raw_pooled_returns_PROCESSED > 0).sum()}, Proportion: {((raw_pooled_returns_PROCESSED > 0).sum() / raw_pooled_returns_PROCESSED.size):.4%}")
print(f"Zero returns:     {(raw_pooled_returns_PROCESSED == 0).sum()}, Proportion: {((raw_pooled_returns_PROCESSED == 0).sum() / raw_pooled_returns_PROCESSED.size):.4%}")
print(f"Negative returns: {(raw_pooled_returns_PROCESSED < 0).sum()}, Proportion: {((raw_pooled_returns_PROCESSED < 0).sum() / raw_pooled_returns_PROCESSED.size):.4%}")

raw_returns_obj shape: (210,) | dtype: object
raw_pooled_returns shape: (2623490,)
Number of zero (0.0%) returns: 200511 (7.6429%)
Positive returns: 1243975, Proportion: 47.4168%
Zero returns:     200511, Proportion: 7.6429%
Negative returns: 1179004, Proportion: 44.9403%


# Raw returns

In [75]:
raw_heavy_prefix = str(RAW_OUTPUT_DIR / "heavy_tailed_distribution")

sf.distribution(
    raw_pooled_returns,
    file_name=raw_heavy_prefix,
    scale="log",
    multiple=False,
    normalize=True,
    granuality=DIST_BINS,
)


In [76]:
raw_vol_prefix = str(RAW_OUTPUT_DIR / "volatility_clustering")

sf.acf(
    raw_returns_obj,
    file_name=raw_vol_prefix,
    for_abs=True,
    multiple=True,
    fit=False,
    scale="log",
    max_lag=MAX_LAG_ACF,
)


In [77]:
raw_lev_prefix = str(RAW_OUTPUT_DIR / "leverage_effect")

lev_values = sf.leverage_effect(
    raw_returns_obj,
    file_name=raw_lev_prefix,
    multiple=True,
    min_lag=1,
    max_lag=MAX_LAG_LEVERAGE,
)

print("L(k) values (lags 1–99):")
print(lev_values)


L(k) values (lags 1–99):
[-10.79876486  -9.14906835  -8.15164988  -6.40271749  -7.30650417
  -5.92523582  -4.99684874  -6.69875218  -4.47797173  -4.64265131
  -4.01478847  -4.35447189  -4.07490293  -3.53601811  -5.01492091
  -1.73285528  -3.96853086  -3.23500055  -1.8708306   -3.58653814
  -4.3723617   -1.89738766  -1.3836941   -2.74780919  -3.43750028
  -2.25131173  -2.25002995  -1.96623491  -2.41750742  -1.93820874
  -2.16453931  -3.33060181  -3.241369    -2.1655992   -2.24747398
  -2.76373384  -1.86596313  -1.59431615  -2.25047069  -2.89433381
  -2.44544758  -1.02225751  -0.06891412  -1.23267503  -0.78221748
  -0.9614069   -1.84036073  -1.84163768  -2.83342909  -0.3468747
  -0.46254363  -1.58859682  -0.62223601  -1.49856748  -0.74576422
  -0.45777854  -1.6508261   -1.22305484  -0.87317721  -1.51261372
  -2.46728233  -0.95346738  -1.36642093  -2.16704487  -1.74128029
  -1.64981209  -1.78072551  -0.88408701  -1.3786373   -1.12870401
  -0.54043048  -2.25175819  -1.59513365  -2.32755568

# Returns OTHER

In [78]:

raw_heavy_prefix = str(RAW_OUTPUT_DIR / "heavy_tailed_distribution_processed")

sf.distribution(
    raw_pooled_returns_PROCESSED,
    file_name=raw_heavy_prefix,
    scale="log",
    multiple=False,
    normalize=True,
    granuality=DIST_BINS,
)


In [79]:
raw_vol_prefix = str(RAW_OUTPUT_DIR / "volatility_clustering_processed")

sf.acf(
    raw_returns_obj_PROCESSED,
    file_name=raw_vol_prefix,
    for_abs=True,
    multiple=True,
    fit=False,
    scale="log",
    max_lag=MAX_LAG_ACF,
)


In [80]:
raw_lev_prefix = str(RAW_OUTPUT_DIR / "leverage_effect_processed")

lev_values = sf.leverage_effect(
    raw_returns_obj_PROCESSED,
    file_name=raw_lev_prefix,
    multiple=True,
    min_lag=1,
    max_lag=MAX_LAG_LEVERAGE,
)

print("L(k) values (lags 1–99):")
print(lev_values)


L(k) values (lags 1–99):
[-10.79876486  -9.14906835  -8.15164988  -6.40271749  -7.30650417
  -5.92523582  -4.99684874  -6.69875218  -4.47797173  -4.64265131
  -4.01478847  -4.35447189  -4.07490293  -3.53601811  -5.01492091
  -1.73285528  -3.96853086  -3.23500055  -1.8708306   -3.58653814
  -4.3723617   -1.89738766  -1.3836941   -2.74780919  -3.43750028
  -2.25131173  -2.25002995  -1.96623491  -2.41750742  -1.93820874
  -2.16453931  -3.33060181  -3.241369    -2.1655992   -2.24747398
  -2.76373384  -1.86596313  -1.59431615  -2.25047069  -2.89433381
  -2.44544758  -1.02225751  -0.06891412  -1.23267503  -0.78221748
  -0.9614069   -1.84036073  -1.84163768  -2.83342909  -0.3468747
  -0.46254363  -1.58859682  -0.62223601  -1.49856748  -0.74576422
  -0.45777854  -1.6508261   -1.22305484  -0.87317721  -1.51261372
  -2.46728233  -0.95346738  -1.36642093  -2.16704487  -1.74128029
  -1.64981209  -1.78072551  -0.88408701  -1.3786373   -1.12870401
  -0.54043048  -2.25175819  -1.59513365  -2.32755568